In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# =========================================================
# SETTINGS
# =========================================================
PRED_PATH = "/content/final_test_predictions.csv"
GOLD_PATH = "/content/test_sen.csv"

ARTICLE_ID = 235
CLAIM_GROUP_ID = None
MAX_TEXT_LEN = None

SAVE_HTML = True
HTML_OUT_PATH = f"/content/article_{ARTICLE_ID}_visual_claim_report.html"

# =========================================================
# LOAD
# =========================================================
pred_df = pd.read_csv(PRED_PATH)
gold_df = pd.read_csv(GOLD_PATH)

for df in [pred_df, gold_df]:
    for col in ["article_id", "claim_group_id", "sentence_id", "sentence_index_in_article"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

pred_df = pred_df.dropna(subset=["article_id", "claim_group_id", "sentence_id"]).copy()
gold_df = gold_df.dropna(subset=["article_id", "claim_group_id", "sentence_id"]).copy()

for df in [pred_df, gold_df]:
    df["article_id"] = df["article_id"].astype(int)
    df["claim_group_id"] = df["claim_group_id"].astype(int)
    df["sentence_id"] = df["sentence_id"].astype(int)

pred_article = pred_df[pred_df["article_id"] == ARTICLE_ID].copy()
gold_article = gold_df[gold_df["article_id"] == ARTICLE_ID].copy()

if pred_article.empty:
    raise ValueError(f"No prediction rows found for article_id={ARTICLE_ID}")

if gold_article.empty:
    raise ValueError(f"No gold rows found for article_id={ARTICLE_ID}")

if CLAIM_GROUP_ID is not None:
    pred_article = pred_article[pred_article["claim_group_id"] == CLAIM_GROUP_ID].copy()
    gold_article = gold_article[gold_article["claim_group_id"] == CLAIM_GROUP_ID].copy()

# =========================================================
# HELPERS
# =========================================================
def shorten(text, max_len=None):
    text = str(text)
    if max_len is None or len(text) <= max_len:
        return text
    return text[:max_len - 3] + "..."

def esc(x):
    return str(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def detect_pred_label_col(df):
    if "pred_role_label" in df.columns:
        return "pred_role_label"
    if "pred_role_label_cv" in df.columns:
        return "pred_role_label_cv"
    if "pred_label" in df.columns:
        return "pred_label"
    return None

def detect_pred_conf_col(df):
    if "pred_confidence" in df.columns:
        return "pred_confidence"
    if "pred_confidence_cv" in df.columns:
        return "pred_confidence_cv"
    return None

def derive_gold_decision(group_df):
    roles = set(group_df["role_label"].astype(str).tolist())
    classes = set(pd.to_numeric(group_df["class_label"], errors="coerce").dropna().astype(int).tolist())

    if "CLAIM" not in roles:
        return "no_claim"
    if 1 in classes:
        return "misleading_false_claim"
    elif 0 in classes:
        return "supported_true_claim"
    else:
        return "unresolved_insufficient_evidence"

def decide_group_hard(group_df):
    pred_label_col = detect_pred_label_col(group_df)
    if pred_label_col is None:
        return "no_claim"

    labels = set(group_df[pred_label_col].astype(str).tolist())

    if "CLAIM" not in labels:
        return "no_claim"

    has_refute = ("VERDICT_REFUTE" in labels) or ("EVIDENCE_REFUTE" in labels)
    has_support = ("VERDICT_SUPPORT" in labels) or ("EVIDENCE_SUPPORT" in labels)

    if has_refute:
        return "misleading_false_claim"
    elif has_support:
        return "supported_true_claim"
    else:
        return "unresolved_insufficient_evidence"

def build_sentence_table(pred_group_df, gold_group_df):
    pred_label_col = detect_pred_label_col(pred_group_df)
    pred_conf_col = detect_pred_conf_col(pred_group_df)

    gold_keep = gold_group_df[[
        "article_id", "claim_group_id", "sentence_id",
        "sentence_index_in_article", "sentence_text", "role_label", "class_label"
    ]].copy()

    pred_keep_cols = [
        "article_id", "claim_group_id", "sentence_id",
        "sentence_index_in_article", "sentence_text"
    ]

    if pred_label_col is not None:
        pred_keep_cols.append(pred_label_col)
    if pred_conf_col is not None:
        pred_keep_cols.append(pred_conf_col)

    pred_keep_cols = [c for c in pred_keep_cols if c in pred_group_df.columns]
    pred_keep = pred_group_df[pred_keep_cols].copy()

    merged = gold_keep.merge(
        pred_keep,
        on=["article_id", "claim_group_id", "sentence_id"],
        how="outer",
        suffixes=("_gold", "_pred")
    ).copy()

    if "sentence_index_in_article_gold" in merged.columns:
        merged["sentence_order"] = pd.to_numeric(merged["sentence_index_in_article_gold"], errors="coerce")
    elif "sentence_index_in_article_pred" in merged.columns:
        merged["sentence_order"] = pd.to_numeric(merged["sentence_index_in_article_pred"], errors="coerce")
    else:
        merged["sentence_order"] = np.arange(len(merged))

    if "sentence_text_gold" in merged.columns:
        text_gold = merged["sentence_text_gold"].fillna("")
    else:
        text_gold = pd.Series([""] * len(merged), index=merged.index)

    if "sentence_text_pred" in merged.columns:
        text_pred = merged["sentence_text_pred"].fillna("")
    else:
        text_pred = pd.Series([""] * len(merged), index=merged.index)

    merged["sentence_text_final"] = np.where(text_gold.astype(str) != "", text_gold, text_pred)
    merged["sentence_text_final"] = merged["sentence_text_final"].astype(str).apply(lambda x: shorten(x, MAX_TEXT_LEN))
    merged["gold_label"] = merged["role_label"].fillna("MISSING").astype(str) if "role_label" in merged.columns else "MISSING"

    if pred_label_col is not None and pred_label_col in merged.columns:
        merged["predicted_label"] = merged[pred_label_col].fillna("MISSING").astype(str)
    else:
        merged["predicted_label"] = "MISSING"

    if pred_conf_col is not None and pred_conf_col in merged.columns:
        merged["pred_confidence_final"] = pd.to_numeric(merged[pred_conf_col], errors="coerce")
    else:
        merged["pred_confidence_final"] = np.nan

    merged["label_correct"] = merged["gold_label"] == merged["predicted_label"]

    out = merged[[
        "article_id", "claim_group_id", "sentence_id", "sentence_order",
        "gold_label", "predicted_label", "label_correct",
        "pred_confidence_final", "sentence_text_final"
    ]].copy()

    out = out.rename(columns={
        "sentence_order": "sentence_index_in_article",
        "pred_confidence_final": "pred_confidence",
        "sentence_text_final": "sentence_text"
    })

    out = out.sort_values("sentence_index_in_article", na_position="last").reset_index(drop=True)
    return out

def role_badge_color(role_name):
    role_name = str(role_name)
    mapping = {
        "BACKGROUND": "#6c757d",
        "CLAIM": "#f59f00",
        "EVIDENCE_SUPPORT": "#2b8a3e",
        "EVIDENCE_REFUTE": "#c92a2a",
        "VERDICT_SUPPORT": "#1c7ed6",
        "VERDICT_REFUTE": "#862e9c",
        "MISSING": "#adb5bd"
    }
    return mapping.get(role_name, "#495057")

def decision_color(decision):
    mapping = {
        "supported_true_claim": "#2f9e44",
        "misleading_false_claim": "#e03131",
        "unresolved_insufficient_evidence": "#f08c00",
        "no_claim": "#6c757d"
    }
    return mapping.get(str(decision), "#495057")

# =========================================================
# BUILD HTML
# =========================================================
pred_label_col_global = detect_pred_label_col(pred_article)
if pred_label_col_global is None:
    raise ValueError("No prediction label column found.")

group_ids = sorted(pred_article["claim_group_id"].unique().tolist())
print(f"Found {len(group_ids)} claim group(s) in article_id={ARTICLE_ID}: {group_ids}")

html_parts = []
html_parts.append("""
<style>
body { font-family: Arial, sans-serif; margin: 24px; }
.report-wrap { max-width: 1200px; margin: 0 auto; }
.group-card {
    border: 1px solid #dee2e6;
    border-radius: 16px;
    padding: 18px 20px;
    margin: 20px 0 28px 0;
    box-shadow: 0 1px 4px rgba(0,0,0,0.06);
}
.group-header { margin-bottom: 12px; }
.group-title { font-size: 22px; font-weight: 700; margin-bottom: 8px; }
.meta-row { display: flex; flex-wrap: wrap; gap: 10px; margin: 8px 0 12px 0; }
.badge {
    display: inline-block;
    padding: 6px 10px;
    border-radius: 999px;
    color: white;
    font-size: 12px;
    font-weight: 700;
}
.summary-box {
    background: #f8f9fa;
    border-radius: 12px;
    padding: 12px 14px;
    margin: 10px 0 16px 0;
}
.sent-list { display: flex; flex-direction: column; gap: 10px; }
.sent-card {
    border-radius: 12px;
    padding: 12px 14px;
    border-left: 8px solid #adb5bd;
}
.sent-card.correct { background: #ebfbee; border-left-color: #2f9e44; }
.sent-card.wrong { background: #fff5f5; border-left-color: #e03131; }
.sent-top { display: flex; flex-wrap: wrap; align-items: center; gap: 8px; margin-bottom: 8px; }
.sent-index { font-weight: 700; color: #343a40; }
.sent-text { font-size: 15px; line-height: 1.55; color: #212529; }
.page-title { font-size: 28px; font-weight: 800; margin-bottom: 20px; }
</style>
""")

html_parts.append('<div class="report-wrap">')
html_parts.append(f'<div class="page-title">Claim-group visual inspection: article_id={ARTICLE_ID}</div>')

for gid in group_ids:
    pred_group = pred_article[pred_article["claim_group_id"] == gid].copy()
    gold_group = gold_article[gold_article["claim_group_id"] == gid].copy()

    if gold_group.empty:
        continue

    gold_decision = derive_gold_decision(gold_group)
    predicted_decision = decide_group_hard(pred_group)
    group_correct = (gold_decision == predicted_decision)

    title = None
    if "title" in gold_group.columns and gold_group["title"].notna().any():
        title = gold_group["title"].dropna().iloc[0]

    sentence_df = build_sentence_table(pred_group, gold_group)

    pred_group_label_col = detect_pred_label_col(pred_group)
    if pred_group_label_col is not None:
        pred_label_set = ", ".join(sorted(set(pred_group[pred_group_label_col].astype(str))))
    else:
        pred_label_set = "MISSING"

    html_parts.append('<div class="group-card">')
    html_parts.append('<div class="group-header">')
    html_parts.append(f'<div class="group-title">Claim group {gid}</div>')
    if title is not None:
        html_parts.append(f'<div><strong>Title:</strong> {esc(title)}</div>')
    html_parts.append('</div>')

    html_parts.append('<div class="meta-row">')
    html_parts.append(f'<span class="badge" style="background:{decision_color(gold_decision)};">Actual: {esc(gold_decision)}</span>')
    html_parts.append(f'<span class="badge" style="background:{decision_color(predicted_decision)};">Predicted: {esc(predicted_decision)}</span>')
    html_parts.append(
        f'<span class="badge" style="background:{"#2f9e44" if group_correct else "#e03131"};">'
        f'{"True" if group_correct else "False"}</span>'
    )
    html_parts.append('</div>')

    html_parts.append('<div class="summary-box">')
    html_parts.append(
        f'<strong>Actual labels in group:</strong> {esc(", ".join(sorted(set(gold_group["role_label"].astype(str)))))}<br>'
    )
    html_parts.append(
        f'<strong>Predicted labels in group:</strong> {esc(pred_label_set)}'
    )
    html_parts.append('</div>')

    html_parts.append('<div class="sent-list">')

    for _, row in sentence_df.iterrows():
        is_correct = bool(row["label_correct"])
        card_class = "correct" if is_correct else "wrong"

        confidence_str = f'{row["pred_confidence"]:.3f}' if pd.notna(row["pred_confidence"]) else "NA"
        idx_value = int(row["sentence_index_in_article"]) if pd.notna(row["sentence_index_in_article"]) else "?"

        html_parts.append(f'<div class="sent-card {card_class}">')
        html_parts.append('<div class="sent-top">')
        html_parts.append(f'<span class="sent-index">[{idx_value}]</span>')
        html_parts.append(f'<span class="badge" style="background:{role_badge_color(row["gold_label"])};">Actual: {esc(row["gold_label"])}</span>')
        html_parts.append(f'<span class="badge" style="background:{role_badge_color(row["predicted_label"])};">Pred: {esc(row["predicted_label"])}</span>')
        html_parts.append(
            f'<span class="badge" style="background:{"#2f9e44" if is_correct else "#e03131"};">'
            f'{"True" if is_correct else "False"}</span>'
        )
        html_parts.append(f'<span class="badge" style="background:#495057;">Conf: {confidence_str}</span>')
        html_parts.append('</div>')
        html_parts.append(f'<div class="sent-text">{esc(row["sentence_text"])}</div>')
        html_parts.append('</div>')

    html_parts.append('</div>')
    html_parts.append('</div>')

html_parts.append('</div>')

full_html = "\n".join(html_parts)

display(HTML(full_html))

if SAVE_HTML:
    with open(HTML_OUT_PATH, "w", encoding="utf-8") as f:
        f.write(full_html)
    print("Saved HTML report to:", HTML_OUT_PATH)

# Auto download
from google.colab import files
files.download(HTML_OUT_PATH)

Found 5 claim group(s) in article_id=235: [0, 1, 2, 3, 4]


Saved HTML report to: /content/article_235_visual_claim_report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>